# Load D5 Dataset


In [2]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import RobustScaler
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

df = pd.read_csv(
    "../datasets/engineered/D5_feature_engineered.csv"
)


print("D5 Dataset Loaded")
print(df.shape)

D5 Dataset Loaded
(119398, 47)


# Drop Raw Datetime Columns and Moonrise/Moonset Removal

In [3]:
df = df.drop(
    columns=[
        "last_updated",
        "sunrise",
        "sunset",
        "moonrise",
        "moonset"
    ]
)
print(df.shape)


(119398, 42)


# Save Scaler

In [4]:
import joblib

# RobustScaler is recommended because EDA showed heavy outliers in PM2.5, PM10, rainfall, and temperature.
scaler = RobustScaler()
joblib.dump(
    scaler,
    "../artifacts/robust_scaler.pkl"
)
# 
print("Scaler Saved")

Scaler Saved


# Scale Features and Save Scaled Dataset

In [5]:
cat_cols = [
    "location_name",
    "region",
    "timezone",
    "condition_text",
    "wind_direction",
    "moon_phase",
    "day_period",
    "season"
]
df_numerical = df.drop(columns=cat_cols)

scaled_df = pd.DataFrame(
    scaler.fit_transform(df_numerical),
    columns=df_numerical.columns
)
scaled_df.to_csv(
    "../datasets/processed/D6_scaled.csv",
    index=False
)

print("Scaled Dataset Saved")

Scaled Dataset Saved


# Create KMeans Dataset

In [6]:
kmeans_df = pd.get_dummies(
    df,
    columns=[
        "location_name",
        "region",
        "timezone",
        "condition_text",
        "wind_direction",
        "moon_phase",
        "day_period",
        "season"
    ],
    drop_first=True
)

print(kmeans_df.shape)
# Save:
kmeans_df.to_csv(
   "../datasets/processed/D6_kmeans_ready.csv",
   index=False
)

print("KMeans Dataset Saved")

(119398, 695)
KMeans Dataset Saved


In [10]:
from sklearn.preprocessing import OneHotEncoder
import pandas as pd
import pickle
import os

# Define columns

cat_cols = [
    "location_name",
    "region",
    "timezone",
    "condition_text",
    "wind_direction",
    "moon_phase",
    "day_period",
    "season"
]

# Fit Encoder

ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

encoded_array = ohe.fit_transform(df[cat_cols])

encoded_df = pd.DataFrame(
    encoded_array,
    columns=ohe.get_feature_names_out(cat_cols)
)

# Merge back numeric columns

kmeans_df = pd.concat(
    [df.drop(columns=cat_cols), encoded_df],
    axis=1
)

# SAVE ONE-HOT ENCODER (.pkl)

# Use a raw string for Windows path to avoid escape-sequence warnings
save_path = r"C:\Users\hp\Documents\ClimateGuardAI\artifacts"
os.makedirs(save_path, exist_ok=True)

with open(os.path.join(save_path, "kmeans_onehot_encoder.pkl"), "wb") as f:
    pickle.dump(ohe, f)

# Save dataset

output_path = "../datasets/processed/D6_kmeans_ready.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
kmeans_df.to_csv(output_path, index=False)


# Final Verification


In [11]:
print("Original Shape:", df.shape)
print("Scaled Shape:", scaled_df.shape)
print(
    scaled_df.head()
)

Original Shape: (119398, 42)
Scaled Shape: (119398, 34)
   latitude  longitude  temperature_celsius  wind_kph  wind_degree  pressure_mb  precip_mm  humidity  cloud  feels_like_celsius  visibility_km  uv_index  gust_kph  air_quality_Carbon_Monoxide  \
0  0.095385  -0.123883             0.759036  2.245902     0.720183    -0.833333        0.0  0.097561  0.450            0.885417            0.0       6.0  0.846847                    -0.664367   
1 -0.095385  -0.113665             0.759036  1.426230     0.747706    -0.833333        0.0  0.170732  0.275            0.916667            0.0       6.0  0.324324                    -0.671068   
2 -0.289231   0.030651             0.614458  1.901639     0.885321    -0.666667        0.0  0.170732  1.075            0.729167            0.0       6.0  0.612613                    -0.710485   
3 -0.321538  -0.097063             0.530120  1.655738     0.793578    -0.666667        0.0  0.317073  1.425            0.666667            0.0       5.0  0.585586  

# Create Preprocessing Report

In [ ]:
import os

# Create folder:
# os.makedirs(
#    "../reports",
#    exist_ok=True
# )
# # Then:
# with open("../reports/preprocessing_report.txt", "w", encoding="utf-8") as f:
os.makedirs(
   "../reports",
   exist_ok=True
)
# Then:
with open("/content/drive/MyDrive/111 - Z-Home - ICTAK-DSA-Z/--- (X) Capstone Project-------------5----/----- 2) Datasets/--->reports/preprocessing_report.txt", "w", encoding="utf-8") as f:
    f.write("""

CLIMATEGUARD AI PREPROCESSING REPORT

D1 Dataset Loading
✓ Completed

D2 Data Quality Verification
✓ Missing Values = 0
✓ Duplicate Records = 0

D3 Feature Removal
✓ Removed 7 redundant features

D4 Datetime Processing
✓ year
✓ month
✓ day
✓ hour
✓ weekday
✓ day_length_minutes
Datetime Cleanup
Moonrise/Moonset Removal

D5 Feature Engineering
✓ temperature_gap
✓ pm_difference
✓ pollution_intensity
✓ wind_humidity_interaction
✓ humidity_cloud_interaction
✓ heatwave_index
✓ day_period
✓ season

D6 Encoding and Scaling
✓ Label Encoding
✓ Robust Scaling

Outlier Handling
✓ Preserved
✓ RobustScaler applied

Feature Selection
✓ Deferred to modelling phase

Generated Datasets
✓ D3_feature_removed.csv
✓ D4_datetime_processed.csv
✓ D5_feature_engineered.csv
✓ D6_label_encoded.csv
✓ D6_scaled.csv
✓ D6_kmeans_ready.csv
Dataset ready for:
1. KMeans Clustering
2. Isolation Forest
3. Rainfall Prediction
4. Heatwave Prediction
5. Climate Risk Engine

Generated Artifacts
✓ label_encoders.pkl
✓ robust_scaler.pkl
""")

print("Preprocessing Report Saved")

Preprocessing Report Saved
